# Aula 21 — Árvores Binárias no SCADA Aqua Index
## BDT Contextual de Intertravamentos & Árvore AVL no Historiador e Catálogo ISA-5.1

---

### 1. Contexto e Escopo Didático

Este *notebook* implementa e valida computacionalmente duas estruturas fundamentais de **Árvores Binárias** aplicadas ao projeto conceitual/simulado **Aqua Index** (Máquina Pneumática de Envasamento de Água em Copos — ECAA08, UNIFEI):

1. **Árvore Binária de Decisão (BDT) Contextual:** Separa **invariantes globais** (`HS-004` e contradição elétrica de fins de curso `R-06`) de **intertravamentos condicionais dependentes do estado da FSM** (`REQ_giro`, `REQ_dose`, `REQ_prensa`, `REQ_braco`, `POS_DISPENSA`). O curto-circuito em $\mathcal{O}(h) = \mathcal{O}(1)$ ($h=7$) arbitra a ação segura determinística, enquanto um **barramento paralelo de alarmes** evita mascaramento de falhas simultâneas na IHM/SCADA.
2. **Árvore Binária de Busca Auto-Balanceada (AVL):** Implementada do zero (com altura, fator de balanceamento $BF = h(\text{dir}) - h(\text{esq})$, rotações `LL`, `RR`, `LR`, `RL`, busca pontual $\mathcal{O}(\log N)$ e consulta por intervalo $[t_{\text{início}}, t_{\text{fim}}]$ em $\mathcal{O}(\log N + K)$ via percurso *in-order* com poda) para:
   - Indexar telemetria **assíncrona / fora de ordem** do Historiador SCADA (eventos **CoS** discretos + leituras periódicas de `TIT-401` em $^\circ\text{C}$ e `SE-001` em graus);
   - Demonstrar consultas ordenadas por faixa de setores sobre as **45 tags ISA-5.1**, contrastando honestamente com o acesso $\mathcal{O}(1)$ de uma tabela *hash* (`dict`).

> **Nota:** Todo o núcleo algorítmico executa integralmente em **Python 3.13 + NumPy**, sem depender obrigatoriamente de `pandas` ou `matplotlib`. Os gráficos na célula final são renderizados automaticamente quando `matplotlib` estiver disponível.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable, Dict, Generic, List, Optional, Tuple, TypeVar
import time
import math
import numpy as np

print(f"Ambiente carregado com sucesso | NumPy {np.__version__}")

---
### 2. Árvore Binária de Decisão (BDT) Contextual da Envasadora

#### Separação de Camadas
- **Nível 0 (Invariante Global):** `HS-004` (`e1 == 1`) $\rightarrow$ `TRIP_EMERGENCIA` (`M-001=0, M-002=0, HT-401=0, IL-002=1`).
- **Nível 1 (Invariante Global — `R-06`):** Contradição elétrica em qualquer par de fins de curso magnéticos ($c_{Xa} \land c_{Xr}$, ex.: `ZSC-401 ∧ ZSO-401`) $\rightarrow$ `TRAVA_SENSOR_MAGNETICO` (`M-001=0`).
- **Nível 2 (FSM — `R-01`):** `REQ_giro ∧ ¬ZSC-401` $\rightarrow$ `BLOQUEIO_COLISAO_MESA` (`M-001=0`).
- **Nível 3 (FSM — `R-05`):** `REQ_dose ∧ ¬ZSC-203` $\rightarrow$ `BLOQUEIO_DOSADOR` (`XV-202=0`).
- **Nível 4 (FSM — `R-02`):** `REQ_prensa ∧ ¬TIT-401` ($T < 180^\circ\text{C}$) $\rightarrow$ `HOLD_ESTAGIO_4` (espera aquecimento; não derruba a máquina inteira).
- **Nível 5 (FSM — `R-03`):** `REQ_braco ∧ ¬PIT-301` $\rightarrow$ `HOLD_ESTAGIO_3` (pausa manipulador).
- **Nível 6 (FSM — `R-04`):** `POS_DISPENSA ∧ ¬ZS-102` $\rightarrow$ `ALARME_REABASTECIMENTO` (pausa ciclo e solicita carga em `MAG-100`, conectando com a Aula 22).
- **Nível 7 (Folha):** `OPERACAO_NORMAL`.

In [ ]:
@dataclass(frozen=True)
class EstadoPlantaAquaIndex:
    '''Snapshot instantâneo dos sensores ISA-5.1 e das solicitações da FSM.'''
    # Invariante global: botão de emergência (Setor 000)
    HS_004: int = 0  # e1: 1 = Emergência pressionada

    # Pares de fins de curso magnéticos (para R-06: contradição cXa ∧ cXr)
    ZSC_101: int = 1; ZSO_101: int = 0  # Cil. A (Setor 100)
    ZSO_202: int = 0; ZSC_202: int = 1  # Cil. C (Setor 200)
    ZSO_301: int = 0; ZSC_301: int = 1  # Cil. D (Setor 300)
    ZSO_302: int = 0; ZSC_302: int = 1  # Cil. E (Setor 300)
    ZSO_401: int = 0; ZSC_401: int = 1  # Cil. F / Prensa (Setor 400): c7r=1 (recuada)
    ZSO_501: int = 0; ZSC_501: int = 1  # Cil. G (Setor 500)
    ZSC_502: int = 1; ZSO_502: int = 0  # Cil. H (Setor 500)

    # Sensores de processo condicionais à fase
    ZSC_203: int = 0          # c4: 1 = Bico aberto confirmado
    TIT_401_temp_c: float = 185.0  # t1 = 1 quando >= 180.0 °C
    PIT_301: int = 0          # p1: 1 = Vácuo / tampa capturada
    ZS_102: int = 1           # s1: 1 = Copo presente em E1

    # Sinais contextuais da Máquina de Estados Finitos (FSM)
    REQ_giro: int = 0         # Solicitação de indexação da mesa (M-001)
    REQ_dose: int = 0         # Solicitação de avanço do dosador (XV-202)
    REQ_prensa: int = 0       # Solicitação de avanço da prensa (XV-401)
    REQ_braco: int = 0        # Solicitação de giro do braço pick-and-place (XV-301)
    POS_DISPENSA: int = 0     # Janela pós-dispensa na Estação 1

    @property
    def TIT_401(self) -> int:
        '''Proposição discreta t1: 1 se T >= 180 °C, 0 caso contrário.'''
        return 1 if self.TIT_401_temp_c >= 180.0 else 0

    def contradicoes_fim_de_curso(self) -> List[str]:
        '''Retorna todos os pares (cXa ∧ cXr) em contradição elétrica simultânea (R-06).'''
        pares = [
            ("Cil. A (ZSC-101 ∧ ZSO-101)", self.ZSC_101 and self.ZSO_101),
            ("Cil. C (ZSO-202 ∧ ZSC-202)", self.ZSO_202 and self.ZSC_202),
            ("Cil. D (ZSO-301 ∧ ZSC-301)", self.ZSO_301 and self.ZSC_301),
            ("Cil. E (ZSO-302 ∧ ZSC-302)", self.ZSO_302 and self.ZSC_302),
            ("Cil. F / Prensa (ZSC-401 ∧ ZSO-401)", self.ZSC_401 and self.ZSO_401),
            ("Cil. G (ZSO-501 ∧ ZSC-501)", self.ZSO_501 and self.ZSC_501),
            ("Cil. H (ZSC-502 ∧ ZSO-502)", self.ZSC_502 and self.ZSO_502),
        ]
        return [nome for nome, ativo in pares if ativo]


@dataclass(frozen=True)
class FolhaAcaoBDT:
    acao: str
    nivel: int
    regra: str
    camada: str
    comandos: Dict[str, Any]
    descricao: str


@dataclass
class NoDecisaoBDT:
    nivel: int
    codigo_alarme: str
    regra: str
    camada: str
    predicado: Callable[[EstadoPlantaAquaIndex], bool]
    ramo_sim: FolhaAcaoBDT
    ramo_nao: "NoDecisaoBDT | FolhaAcaoBDT"


@dataclass
class ResultadoAvaliacaoBDT:
    folha_arbitrada: FolhaAcaoBDT
    passos_curto_circuito: int
    alarmes_simultaneos: List[str]
    detalhes_contradicao_r06: List[str]


class BDTContextualAquaIndex:
    '''
    Árvore Binária de Decisão (BDT) Contextual de altura fixa h = 7.
    - Avaliação por curto-circuito O(h) = O(1) para arbitrar a ação segura única.
    - Barramento paralelo de alarmes para registrar todas as falhas simultâneas sem mascaramento.
    '''
    ALTURA_FIXA: int = 7

    def __init__(self) -> None:
        folha_normal = FolhaAcaoBDT(
            acao="OPERACAO_NORMAL",
            nivel=7,
            regra="NOMINAL",
            camada="Nominal",
            comandos={"M-001": "LIBERADO", "M-002": "LIBERADO", "HT-401": "CONTROLE_TEMP", "IL-002": 0},
            descricao="Todos os invariantes globais e intertravamentos da FSM satisfeitos.",
        )

        # Construção bottom-up (do Nível 6 até a Raiz Nível 0)
        n6 = NoDecisaoBDT(
            nivel=6,
            codigo_alarme="ALM-R04: POS_DISPENSA ∧ ¬ZS-102 (MAG-100 vazio / falta de copo E1)",
            regra="R-04",
            camada="Intertravamento FSM",
            predicado=lambda s: bool(s.POS_DISPENSA and not s.ZS_102),
            ramo_sim=FolhaAcaoBDT(
                acao="ALARME_REABASTECIMENTO",
                nivel=6,
                regra="R-04",
                camada="Intertravamento FSM",
                comandos={"M-001": 0, "MAG-100": "SOLICITAR_REABASTECIMENTO", "IL-002": 0},
                descricao="Pausa ciclo de indexação e solicita reabastecimento de MAG-100 (Aula 22).",
            ),
            ramo_nao=folha_normal,
        )

        n5 = NoDecisaoBDT(
            nivel=5,
            codigo_alarme="ALM-R03: REQ_braco ∧ ¬PIT-301 (Vácuo não confirmado em E3)",
            regra="R-03",
            camada="Intertravamento FSM",
            predicado=lambda s: bool(s.REQ_braco and not s.PIT_301),
            ramo_sim=FolhaAcaoBDT(
                acao="HOLD_ESTAGIO_3",
                nivel=5,
                regra="R-03",
                camada="Intertravamento FSM",
                comandos={"XV-301": 0, "IL-002": 0},
                descricao="Pausa manipulador pick-and-place em E3 até confirmação de vácuo PIT-301.",
            ),
            ramo_nao=n6,
        )

        n4 = NoDecisaoBDT(
            nivel=4,
            codigo_alarme="ALM-R02: REQ_prensa ∧ ¬TIT-401 (T < 180 °C em E4)",
            regra="R-02",
            camada="Intertravamento FSM",
            predicado=lambda s: bool(s.REQ_prensa and not s.TIT_401),
            ramo_sim=FolhaAcaoBDT(
                acao="HOLD_ESTAGIO_4",
                nivel=4,
                regra="R-02",
                camada="Intertravamento FSM",
                comandos={"XV-401": 0, "HT-401": 1, "M-002": "MANTIDO", "IL-002": 0},
                descricao="Espera aquecimento em E4 (T >= 180 °C) sem derrubar a máquina inteira.",
            ),
            ramo_nao=n5,
        )

        n3 = NoDecisaoBDT(
            nivel=3,
            codigo_alarme="ALM-R05: REQ_dose ∧ ¬ZSC-203 (Bico não confirmado aberto em E2)",
            regra="R-05",
            camada="Intertravamento FSM",
            predicado=lambda s: bool(s.REQ_dose and not s.ZSC_203),
            ramo_sim=FolhaAcaoBDT(
                acao="BLOQUEIO_DOSADOR",
                nivel=3,
                regra="R-05",
                camada="Intertravamento FSM",
                comandos={"XV-202": 0, "IL-002": 0},
                descricao="Aborta avanço do cilindro dosador XV-202 para evitar sobrepressão.",
            ),
            ramo_nao=n4,
        )

        n2 = NoDecisaoBDT(
            nivel=2,
            codigo_alarme="ALM-R01: REQ_giro ∧ ¬ZSC-401 (Prensa não recuada ao girar mesa)",
            regra="R-01",
            camada="Intertravamento FSM",
            predicado=lambda s: bool(s.REQ_giro and not s.ZSC_401),
            ramo_sim=FolhaAcaoBDT(
                acao="BLOQUEIO_COLISAO_MESA",
                nivel=2,
                regra="R-01",
                camada="Intertravamento FSM",
                comandos={"M-001": 0, "IL-002": 0},
                descricao="Inibe motor da mesa indexadora M-001 por risco de colisão mecânica.",
            ),
            ramo_nao=n3,
        )

        n1 = NoDecisaoBDT(
            nivel=1,
            codigo_alarme="ALM-R06: Contradição elétrica de fins de curso (cXa ∧ cXr)",
            regra="R-06",
            camada="Invariante Global",
            predicado=lambda s: len(s.contradicoes_fim_de_curso()) > 0,
            ramo_sim=FolhaAcaoBDT(
                acao="TRAVA_SENSOR_MAGNETICO",
                nivel=1,
                regra="R-06",
                camada="Invariante Global",
                comandos={"M-001": 0, "MODO_AUTO": 0, "IL-002": 0},
                descricao="Trava modo automático por incoerência elétrica em sensor magnético.",
            ),
            ramo_nao=n2,
        )

        self.raiz = NoDecisaoBDT(
            nivel=0,
            codigo_alarme="ALM-EMERG: HS-004 (e1 == 1 — Botão de Emergência acionado)",
            regra="EMERG-00",
            camada="Invariante Global",
            predicado=lambda s: bool(s.HS_004 == 1),
            ramo_sim=FolhaAcaoBDT(
                acao="TRIP_EMERGENCIA",
                nivel=0,
                regra="EMERG-00",
                camada="Invariante Global",
                comandos={"M-001": 0, "M-002": 0, "HT-401": 0, "IL-002": 1},
                descricao="Parada imediata geral: desliga mesa, esteira e resistência; acende IL-002.",
            ),
            ramo_nao=n1,
        )

        # Lista ordenada dos 7 nós internos para varredura do barramento paralelo de alarmes
        self._nos_barramento: List[NoDecisaoBDT] = [self.raiz, n1, n2, n3, n4, n5, n6]

    def avaliar(self, estado: EstadoPlantaAquaIndex) -> ResultadoAvaliacaoBDT:
        # 1) Curto-circuito na BDT para ação segura determinística O(h) = O(1)
        atual: NoDecisaoBDT | FolhaAcaoBDT = self.raiz
        passos = 0
        while isinstance(atual, NoDecisaoBDT):
            passos += 1
            if atual.predicado(estado):
                atual = atual.ramo_sim
            else:
                atual = atual.ramo_nao

        # 2) Barramento paralelo de alarmes (sem mascaramento na IHM/SCADA)
        alarmes_ativos = [no.codigo_alarme for no in self._nos_barramento if no.predicado(estado)]
        return ResultadoAvaliacaoBDT(
            folha_arbitrada=atual,
            passos_curto_circuito=passos,
            alarmes_simultaneos=alarmes_ativos,
            detalhes_contradicao_r06=estado.contradicoes_fim_de_curso(),
        )


# Validação dos 8 cenários + cenários de contextualidade (falsos positivos evitados) e multi-falha
bdt = BDTContextualAquaIndex()

cenarios_bdt: List[Tuple[str, EstadoPlantaAquaIndex, str, int]] = [
    ("C0 - Emergência HS-004 (Invariante Global)", EstadoPlantaAquaIndex(HS_004=1), "TRIP_EMERGENCIA", 1),
    ("C1 - Contradição Prensa ZSC-401 ∧ ZSO-401 (R-06)", EstadoPlantaAquaIndex(ZSC_401=1, ZSO_401=1), "TRAVA_SENSOR_MAGNETICO", 2),
    ("C2 - Giro com prensa avançada REQ_giro ∧ ¬ZSC-401 (R-01)", EstadoPlantaAquaIndex(REQ_giro=1, ZSC_401=0, ZSO_401=1), "BLOQUEIO_COLISAO_MESA", 3),
    ("C3 - Dosagem com bico fechado REQ_dose ∧ ¬ZSC-203 (R-05)", EstadoPlantaAquaIndex(REQ_dose=1, ZSC_203=0), "BLOQUEIO_DOSADOR", 4),
    ("C4 - Prensa fria REQ_prensa ∧ T=162°C (R-02)", EstadoPlantaAquaIndex(REQ_prensa=1, TIT_401_temp_c=162.0), "HOLD_ESTAGIO_4", 5),
    ("C5 - Giro do braço sem vácuo REQ_braco ∧ ¬PIT-301 (R-03)", EstadoPlantaAquaIndex(REQ_braco=1, PIT_301=0), "HOLD_ESTAGIO_3", 6),
    ("C6 - Falta de copo pós-dispensa POS_DISPENSA ∧ ¬ZS-102 (R-04)", EstadoPlantaAquaIndex(POS_DISPENSA=1, ZS_102=0), "ALARME_REABASTECIMENTO", 7),
    ("C7 - Operação Normal (sensores fora de fase em 0 NÃO disparam trip)", EstadoPlantaAquaIndex(
        ZSC_203=0, PIT_301=0, TIT_401_temp_c=150.0, ZS_102=0,
        REQ_giro=0, REQ_dose=0, REQ_prensa=0, REQ_braco=0, POS_DISPENSA=0
    ), "OPERACAO_NORMAL", 7),
]

print(f"{'Cenário':<64} | {'Ação Arbitrada':<24} | {'Passos':<6} | {'Alarmes'}")
print("-" * 112)
for nome, est, esperado, passos_esp in cenarios_bdt:
    res = bdt.avaliar(est)
    assert res.folha_arbitrada.acao == esperado, f"Falha em {nome}: {res.folha_arbitrada.acao} != {esperado}"
    assert res.passos_curto_circuito == passos_esp, f"Passos inesperados em {nome}: {res.passos_curto_circuito}"
    assert res.passos_curto_circuito <= bdt.ALTURA_FIXA
    print(f"{nome:<64} | {res.folha_arbitrada.acao:<24} | {res.passos_curto_circuito:<6} | {len(res.alarmes_simultaneos)}")

# Teste de Múltiplas Falhas Simultâneas (Curto-circuito arbitra + Barramento não mascara)
estado_multifalha = EstadoPlantaAquaIndex(
    HS_004=1,
    REQ_giro=1, ZSC_401=0, ZSO_401=1,     # R-01 ativa também
    REQ_prensa=1, TIT_401_temp_c=170.5,   # R-02 ativa também
    POS_DISPENSA=1, ZS_102=0,             # R-04 ativa também
)
res_multi = bdt.avaliar(estado_multifalha)
assert res_multi.folha_arbitrada.acao == "TRIP_EMERGENCIA"
assert res_multi.passos_curto_circuito == 1
assert len(res_multi.alarmes_simultaneos) == 4
print("\n[Cenário Multi-Falha Simultânea]")
print(f"  Ação segura arbitrada em {res_multi.passos_curto_circuito} passo (curto-circuito): {res_multi.folha_arbitrada.acao}")
print(f"  Comandos enviados aos atuadores: {res_multi.folha_arbitrada.comandos}")
print(f"  Alarmes simultâneos preservados no barramento SCADA ({len(res_multi.alarmes_simultaneos)}):")
for alm in res_multi.alarmes_simultaneos:
    print(f"    - {alm}")

---
### 3. Implementação do Zero da Árvore AVL (`ArvoreAVLHistoriador`)

A classe `ArvoreAVL` abaixo é genérica (suportando tanto *timestamps* do Historiador quanto chaves compostas `(setor, tag)` do catálogo ISA-5.1) e implementa:
- Altura $h(n) = 1 + \max(h(\text{esq}), h(\text{dir}))$, com $h(\text{None}) = 0$;
- Fator de balanceamento $BF(n) = h(\text{dir}) - h(\text{esq}) \in \{-1, 0, +1\}$;
- Rotações **LL**, **RR**, **LR** e **RL** em $\mathcal{O}(1)$ com telemetria de contagem;
- Busca pontual em $\mathcal{O}(\log N)$;
- Consulta por intervalo $[k_{\min}, k_{\max}]$ em $\mathcal{O}(\log N + K)$ via percurso *in-order* com poda de subárvores;
- **Verificador formal da invariante AVL** (`validar_invariante()`), checando ordenação BST, consistência exata das alturas e $|BF(n)| \le 1$ em todos os nós.

In [ ]:
K = TypeVar("K")
V = TypeVar("V")


@dataclass
class NoAVL(Generic[K, V]):
    chave: K
    payload: V
    altura: int = 1
    esq: Optional["NoAVL[K, V]"] = None
    dir: Optional["NoAVL[K, V]"] = None


class ArvoreAVLHistoriador(Generic[K, V]):
    '''Árvore AVL implementada do zero com BF = h(dir) - h(esq) e consulta de intervalo in-order.'''

    def __init__(self) -> None:
        self.raiz: Optional[NoAVL[K, V]] = None
        self.tamanho: int = 0
        self.contagem_rotacoes: Dict[str, int] = {"LL": 0, "RR": 0, "LR": 0, "RL": 0}

    @staticmethod
    def _altura(no: Optional[NoAVL[K, V]]) -> int:
        return no.altura if no is not None else 0

    def _atualizar_altura(self, no: NoAVL[K, V]) -> None:
        no.altura = 1 + max(self._altura(no.esq), self._altura(no.dir))

    def _fator_balanceamento(self, no: Optional[NoAVL[K, V]]) -> int:
        if no is None:
            return 0
        return self._altura(no.dir) - self._altura(no.esq)

    def _rotacao_direita(self, y: NoAVL[K, V]) -> NoAVL[K, V]:
        x = y.esq
        assert x is not None
        t2 = x.dir
        x.dir = y
        y.esq = t2
        self._atualizar_altura(y)
        self._atualizar_altura(x)
        return x

    def _rotacao_esquerda(self, x: NoAVL[K, V]) -> NoAVL[K, V]:
        y = x.dir
        assert y is not None
        t2 = y.esq
        y.esq = x
        x.dir = t2
        self._atualizar_altura(x)
        self._atualizar_altura(y)
        return y

    def inserir(self, chave: K, payload: V) -> None:
        self.raiz = self._inserir_rec(self.raiz, chave, payload)

    def _inserir_rec(self, no: Optional[NoAVL[K, V]], chave: K, payload: V) -> NoAVL[K, V]:
        if no is None:
            self.tamanho += 1
            return NoAVL(chave=chave, payload=payload)

        if chave < no.chave:  # type: ignore[operator]
            no.esq = self._inserir_rec(no.esq, chave, payload)
        elif chave > no.chave:  # type: ignore[operator]
            no.dir = self._inserir_rec(no.dir, chave, payload)
        else:
            # Atualização idempotente para mesma chave
            no.payload = payload
            return no

        self._atualizar_altura(no)
        bf = self._fator_balanceamento(no)

        # Caso LL: subárvore esquerda mais alta e pendente à esquerda
        if bf < -1 and self._fator_balanceamento(no.esq) <= 0:
            self.contagem_rotacoes["LL"] += 1
            return self._rotacao_direita(no)

        # Caso LR: subárvore esquerda mais alta e pendente à direita
        if bf < -1 and self._fator_balanceamento(no.esq) > 0:
            self.contagem_rotacoes["LR"] += 1
            assert no.esq is not None
            no.esq = self._rotacao_esquerda(no.esq)
            return self._rotacao_direita(no)

        # Caso RR: subárvore direita mais alta e pendente à direita
        if bf > 1 and self._fator_balanceamento(no.dir) >= 0:
            self.contagem_rotacoes["RR"] += 1
            return self._rotacao_esquerda(no)

        # Caso RL: subárvore direita mais alta e pendente à esquerda
        if bf > 1 and self._fator_balanceamento(no.dir) < 0:
            self.contagem_rotacoes["RL"] += 1
            assert no.dir is not None
            no.dir = self._rotacao_direita(no.dir)
            return self._rotacao_esquerda(no)

        return no

    def buscar(self, chave: K) -> Tuple[Optional[V], int]:
        '''Busca pontual O(log N). Retorna (payload_ou_None, comparacoes).'''
        atual = self.raiz
        comparacoes = 0
        while atual is not None:
            comparacoes += 1
            if chave == atual.chave:
                return atual.payload, comparacoes
            if chave < atual.chave:  # type: ignore[operator]
                atual = atual.esq
            else:
                atual = atual.dir
        return None, comparacoes

    def consultar_intervalo(self, k_inicio: K, k_fim: K) -> List[Tuple[K, V]]:
        '''Consulta ordenada por intervalo [k_inicio, k_fim] em O(log N + K) via in-order com poda.'''
        resultado: List[Tuple[K, V]] = []

        def _inorder_podado(no: Optional[NoAVL[K, V]]) -> None:
            if no is None:
                return
            if no.chave > k_inicio:  # type: ignore[operator]
                _inorder_podado(no.esq)
            if k_inicio <= no.chave <= k_fim:  # type: ignore[operator]
                resultado.append((no.chave, no.payload))
            if no.chave < k_fim:  # type: ignore[operator]
                _inorder_podado(no.dir)

        _inorder_podado(self.raiz)
        return resultado

    def altura_total(self) -> int:
        return self._altura(self.raiz)

    def validar_invariante(self) -> Dict[str, Any]:
        '''Verifica formalmente em TODOS os nós: ordem BST, altura exata e |BF| <= 1.'''
        nos_visitados = 0
        max_abs_bf = 0
        chave_anterior: List[Any] = [None]

        def _verificar(no: Optional[NoAVL[K, V]]) -> int:
            nonlocal nos_visitados, max_abs_bf
            if no is None:
                return 0
            h_esq = _verificar(no.esq)
            if chave_anterior[0] is not None:
                assert no.chave > chave_anterior[0], f"Violação BST: {no.chave} <= {chave_anterior[0]}"
            chave_anterior[0] = no.chave
            h_dir = _verificar(no.dir)

            h_real = 1 + max(h_esq, h_dir)
            assert no.altura == h_real, f"Altura inconsistente em {no.chave}: {no.altura} != {h_real}"
            bf = h_dir - h_esq
            assert -1 <= bf <= 1, f"Violação AVL em {no.chave}: BF={bf}"
            max_abs_bf = max(max_abs_bf, abs(bf))
            nos_visitados += 1
            return h_real

        altura_verificada = _verificar(self.raiz)
        assert nos_visitados == self.tamanho, f"Contagem inconsistente: {nos_visitados} != {self.tamanho}"
        return {
            "valido": True,
            "nos": nos_visitados,
            "altura": altura_verificada,
            "max_abs_bf": max_abs_bf,
            "rotacoes": dict(self.contagem_rotacoes),
        }


# Teste unitário explícito das 4 rotações canônicas (LL, RR, LR, RL)
for caso, seq in [
    ("LL", [30, 20, 10]),
    ("RR", [10, 20, 30]),
    ("LR", [30, 10, 20]),
    ("RL", [10, 30, 20]),
]:
    t = ArvoreAVLHistoriador[int, str]()
    for k in seq:
        t.inserir(k, f"ev_{k}")
    inv = t.validar_invariante()
    assert inv["rotacoes"][caso] == 1, f"Rotação {caso} não acionada corretamente: {inv['rotacoes']}"
    assert t.raiz is not None and t.raiz.chave == 20
print("Testes unitários das 4 rotações AVL (LL, RR, LR, RL) validados com sucesso!")

---
### 4. Simulação de Telemetria SCADA Assíncrona (CoS + Analógica Fora de Ordem)

Neste experimento, simulamos um trecho de operação do Historiador **Aqua Index** contendo:
1. **Leituras periódicas analógicas** (`TIT-401` em $^\circ\text{C}$ e `SE-001` em graus de $0^\circ$ a $360^\circ$);
2. **Eventos discretos por Mudança de Estado (CoS)** dos sensores e da BDT (coerente com a política anti-redundância da Aula 20);
3. **Ações do operador na IHM** (`HS-001`, `HS-003`) e **rajada de reconexão** após microqueda de rede Cliente-Servidor, causando chegada **fora de ordem temporal** ($\tau_k < \tau_{k-1}$).

Em seguida, executamos uma **consulta por intervalo $[t_{\text{início}}, t_{\text{fim}}]$** para Análise de Causa-Raiz (RCA) ao redor de uma queda térmica simulada (`TIT-401 < 180 °C` $\rightarrow$ `HOLD_ESTAGIO_4`).

In [ ]:
@dataclass(frozen=True)
class RegistroTelemetriaSCADA:
    timestamp_ms: int
    tipo_evento: str        # "CoS", "ANALOG" ou "IHM"
    origem: str             # "CLP_AS", "IHM_OP" ou "BUFFER_REENVIO"
    tag: str
    valor: Any
    tit_401_c: float
    se_001_deg: float
    acao_bdt: str


def gerar_telemetria_fora_de_ordem(n_ciclos_analog: int = 1200, seed: int = 42) -> List[RegistroTelemetriaSCADA]:
    '''
    Gera série realista do Historiador Aqua Index:
    - Amostras analógicas periódicas (a cada 100 ms): TIT-401 (°C) e SE-001 (graus).
    - Eventos CoS discretos (apenas quando há mudança de estado).
    - Eventos IHM + jitter de rede + rajada de reconexão (chegada fora de ordem).
    '''
    rng = np.random.default_rng(seed)
    t0 = 1_772_450_000_000  # Timestamp base em ms
    eventos_cronologicos: List[RegistroTelemetriaSCADA] = []

    angulo = 0.0
    temp_base = 186.0
    ultimo_estado_bdt = "OPERACAO_NORMAL"

    for i in range(n_ciclos_analog):
        ts = t0 + i * 100
        # Mesa indexa 72° (360° / 5 estações) periodicamente
        if i % 10 == 0:
            angulo = (angulo + 72.0) % 360.0

        # Simula perturbação térmica (queda abaixo de 180 °C) entre i=520 e i=545
        if 520 <= i <= 545:
            temp = 175.0 + float(rng.normal(0.0, 0.8))
        else:
            temp = temp_base + float(rng.normal(0.0, 0.6))

        # Avaliação contextual BDT na Estação 4 durante janela de prensagem
        req_prensa = 1 if (i % 10 in (3, 4, 5)) else 0
        est = EstadoPlantaAquaIndex(TIT_401_temp_c=temp, REQ_prensa=req_prensa)
        acao = bdt.avaliar(est).folha_arbitrada.acao

        # 1) Amostra periódica analógica (TIT-401 e SE-001)
        eventos_cronologicos.append(
            RegistroTelemetriaSCADA(
                timestamp_ms=ts,
                tipo_evento="ANALOG",
                origem="CLP_AS",
                tag="TIT-401/SE-001",
                valor={"TIT-401": round(temp, 2), "SE-001": round(angulo, 1)},
                tit_401_c=round(temp, 2),
                se_001_deg=round(angulo, 1),
                acao_bdt=acao,
            )
        )

        # 2) Evento discreto CoS somente quando o estado muda (anti-redundância, Aula 20)
        if acao != ultimo_estado_bdt:
            eventos_cronologicos.append(
                RegistroTelemetriaSCADA(
                    timestamp_ms=ts + 7,
                    tipo_evento="CoS",
                    origem="CLP_AS",
                    tag="TIT-401" if acao == "HOLD_ESTAGIO_4" else "BDT-CORE",
                    valor=acao,
                    tit_401_c=round(temp, 2),
                    se_001_deg=round(angulo, 1),
                    acao_bdt=acao,
                )
            )
            ultimo_estado_bdt = acao

        # 3) Eventos pontuais CoS de sensores de processo e IHM
        if i == 300:
            eventos_cronologicos.append(
                RegistroTelemetriaSCADA(
                    timestamp_ms=ts + 13,
                    tipo_evento="IHM",
                    origem="IHM_OP",
                    tag="HS-001",
                    valor=1,
                    tit_401_c=round(temp, 2),
                    se_001_deg=round(angulo, 1),
                    acao_bdt="OPERACAO_NORMAL",
                )
            )
        if i == 530:
            # Simula evento CoS atrasado de falta de copo em E1 (R-04) vindo de buffer
            eventos_cronologicos.append(
                RegistroTelemetriaSCADA(
                    timestamp_ms=ts + 21,
                    tipo_evento="CoS",
                    origem="BUFFER_REENVIO",
                    tag="ZS-102",
                    valor=0,
                    tit_401_c=round(temp, 2),
                    se_001_deg=round(angulo, 1),
                    acao_bdt="ALARME_REABASTECIMENTO",
                )
            )

    # Simulação de chegada fora de ordem (jitter de rede + rajada de reconexão)
    ordem_chegada = list(eventos_cronologicos)
    for start in range(0, len(ordem_chegada) - 6, 6):
        bloco = ordem_chegada[start : start + 6]
        perm = rng.permutation(len(bloco))
        ordem_chegada[start : start + 6] = [bloco[j] for j in perm]

    # Rajada de reconexão: desloca um pacote de 25 eventos antigos para chegar mais tarde
    bloco_atrasado = ordem_chegada[510:535]
    del ordem_chegada[510:535]
    ordem_chegada[650:650] = bloco_atrasado

    return ordem_chegada


fluxo_chegada = gerar_telemetria_fora_de_ordem(n_ciclos_analog=1200, seed=42)
inversoes = sum(
    1 for a, b in zip(fluxo_chegada, fluxo_chegada[1:]) if b.timestamp_ms < a.timestamp_ms
)

historiador_avl = ArvoreAVLHistoriador[int, RegistroTelemetriaSCADA]()
for ev in fluxo_chegada:
    historiador_avl.inserir(ev.timestamp_ms, ev)

invariante_hist = historiador_avl.validar_invariante()
print("=== Historiador SCADA Aqua Index (Árvore AVL) ===")
print(f"Total de eventos ingeridos     : {historiador_avl.tamanho}")
print(f"Inversões temporais na chegada : {inversoes} (fluxo assíncrono / fora de ordem)")
print(f"Altura AVL resultante          : {invariante_hist['altura']} (limite teórico <= {math.ceil(1.44 * math.log2(historiador_avl.tamanho + 2))})")
print(f"Máximo |BF| em todos os nós    : {invariante_hist['max_abs_bf']} (invariante válida: {invariante_hist['valido']})")
print(f"Rotações executadas            : {invariante_hist['rotacoes']}")

# Consulta por intervalo [t1, t2] na janela da perturbação térmica (RCA)
t_base = 1_772_450_000_000
t_ini = t_base + 520 * 100
t_fim = t_base + 532 * 100
janela_rca = historiador_avl.consultar_intervalo(t_ini, t_fim)

# Validação formal contra força bruta ordenada
ref_forca_bruta = sorted(
    [(e.timestamp_ms, e) for e in fluxo_chegada if t_ini <= e.timestamp_ms <= t_fim],
    key=lambda x: x[0],
)
assert [k for k, _ in janela_rca] == [k for k, _ in ref_forca_bruta], "Divergência na consulta de intervalo!"
assert all(a[0] < b[0] for a, b in zip(janela_rca, janela_rca[1:])), "Saída in-order não está estritamente ordenada!"

print(f"\n=== Consulta por Intervalo RCA [{t_ini}, {t_fim}] -> {len(janela_rca)} eventos recuperados em ordem ===")
print(f"{'Δt (ms)':>8} | {'Tipo':<6} | {'Origem':<14} | {'Tag':<14} | {'TIT-401':>8} | {'SE-001':>6} | {'Ação BDT'}")
print("-" * 96)
for ts, reg in janela_rca[:12]:
    print(
        f"{ts - t_base:>8} | {reg.tipo_evento:<6} | {reg.origem:<14} | {reg.tag:<14} | "
        f"{reg.tit_401_c:>6.2f}°C | {reg.se_001_deg:>5.1f}° | {reg.acao_bdt}"
    )
if len(janela_rca) > 12:
    print(f"... (+ {len(janela_rca) - 12} registros omitidos na prévia)")

---
### 5. Benchmark Empírico: Busca AVL $\mathcal{O}(\log N)$ vs. Busca Linear $\mathcal{O}(N)$

Comparamos o tempo médio e o número de comparações da busca pontual por *timestamp* na **Árvore AVL** contra a **busca linear** em um vetor não ordenado de eventos assíncronos para diferentes volumes $N$ do Historiador.

In [ ]:
tamanhos_n = [1_000, 3_000, 10_000, 25_000]
n_consultas = 400
rng_bench = np.random.default_rng(2026)

resultados_bench: List[Dict[str, float]] = []

print(f"{'N':>8} | {'Altura AVL':>10} | {'Comp. AVL':>10} | {'Comp. Linear':>12} | {'Tempo AVL (ms)':>14} | {'Tempo Linear (ms)':>17} | {'Speedup':>8}")
print("-" * 96)

for n in tamanhos_n:
    chaves = np.arange(n, dtype=np.int64) * 10
    chaves_embaralhadas = rng_bench.permutation(chaves)

    avl_b = ArvoreAVLHistoriador[int, int]()
    lista_raw: List[Tuple[int, int]] = []
    for k in chaves_embaralhadas:
        ki = int(k)
        avl_b.inserir(ki, ki)
        lista_raw.append((ki, ki))

    # Valida invariante AVL para cada N testado
    inv_b = avl_b.validar_invariante()
    assert inv_b["valido"] and inv_b["max_abs_bf"] <= 1

    alvos = [int(x) for x in rng_bench.choice(chaves, size=n_consultas, replace=False)]

    # Benchmark AVL
    t0 = time.perf_counter()
    comp_avl_total = 0
    for alvo in alvos:
        val, comps = avl_b.buscar(alvo)
        assert val == alvo
        comp_avl_total += comps
    dt_avl_ms = (time.perf_counter() - t0) * 1000.0

    # Benchmark Busca Linear O(N)
    t0 = time.perf_counter()
    comp_lin_total = 0
    for alvo in alvos:
        achou = None
        for idx, (k_item, v_item) in enumerate(lista_raw, start=1):
            if k_item == alvo:
                achou = v_item
                comp_lin_total += idx
                break
        assert achou == alvo
    dt_lin_ms = (time.perf_counter() - t0) * 1000.0

    comp_avl_med = comp_avl_total / n_consultas
    comp_lin_med = comp_lin_total / n_consultas
    speedup = dt_lin_ms / max(dt_avl_ms, 1e-9)

    resultados_bench.append(
        {
            "N": float(n),
            "altura": float(inv_b["altura"]),
            "comp_avl": comp_avl_med,
            "comp_linear": comp_lin_med,
            "tempo_avl_ms": dt_avl_ms,
            "tempo_linear_ms": dt_lin_ms,
            "speedup": speedup,
        }
    )
    print(
        f"{n:>8} | {inv_b['altura']:>10} | {comp_avl_med:>10.1f} | {comp_lin_med:>12.1f} | "
        f"{dt_avl_ms:>14.3f} | {dt_lin_ms:>17.3f} | {speedup:>7.1f}x"
    )

# Asserção de ganho assintótico: para N=25.000, AVL faz ordens de grandeza menos comparações
assert resultados_bench[-1]["comp_avl"] < 20.0
assert resultados_bench[-1]["comp_linear"] > 5_000.0
assert resultados_bench[-1]["speedup"] > 20.0

---
### 6. Aplicação Complementar: Catálogo das 45 Tags ISA-5.1 em AVL (Consulta por Faixa de Setores)

Abaixo cadastramos exatamente as **45 tags ISA-5.1** de `etapa-01-logica/02 - Mapeamento_de_variaveis.md` em duas estruturas:
- Uma tabela *hash* (`dict`) — ideal para **busca exata** por tag em $\mathcal{O}(1)$;
- Uma **Árvore AVL** indexada pela chave composta `(setor, tag)` — que viabiliza **consultas ordenadas por faixa de setores** (ex.: todas as tags entre Setor 200 e Setor 400 em ordem lexicográfica por setor) em $\mathcal{O}(\log N + K)$.

In [ ]:
CATALOGO_45_TAGS_ISA51: List[Tuple[int, str, str, str]] = [
    # Setor 000: Controle Geral (11 tags)
    (0, "M-001", "m0", "Motor da mesa indexadora"),
    (0, "M-002", "m1", "Motor da esteira de saída"),
    (0, "SE-001", "p0", "Sensor de posição (Encoder)"),
    (0, "HS-001", "b1", "Botão Ligar"),
    (0, "HS-002", "b2", "Botão Desligar"),
    (0, "HS-003", "b3", "Botão Parar"),
    (0, "HS-004", "e1", "Botão Emergência"),
    (0, "HS-005", "k1", "Chave Energizada"),
    (0, "IL-001", "l1", "LED Ligado"),
    (0, "IL-002", "l2", "LED Emergência"),
    (0, "IL-003", "l3", "LED Energizado"),
    # Setor 100: Dispensa de Copo (4 tags)
    (100, "XV-101", "v1", "Válvula (Cilindro A)"),
    (100, "ZSC-101", "c1a", "Sensor Mag. (Cil. A Avanço)"),
    (100, "ZSO-101", "c1r", "Sensor Mag. (Cil. A Recuo)"),
    (100, "ZS-102", "s1", "Sensor Capacitivo E1"),
    # Setor 200: Envase de Água (8 tags)
    (200, "ZS-200", "s2", "Sensor Capacitivo E2"),
    (200, "XV-201", "v2", "Válvula 3 vias (Cilindro B)"),
    (200, "ZSC-201", "c2", "Sensor Mag. (Cil. B)"),
    (200, "XV-202", "v3", "Dosador (Cilindro C)"),
    (200, "ZSC-202", "c3r", "Sensor Mag. (Cil. C Recuo)"),
    (200, "ZSO-202", "c3a", "Sensor Mag. (Cil. C Avanço)"),
    (200, "XV-203", "v4", "Válvula do Bico"),
    (200, "ZSC-203", "c4", "Sensor Mag. (Bico)"),
    # Setor 300: Pick-and-Place / Tampa (9 tags)
    (300, "ZS-300", "s3", "Sensor Capacitivo E3"),
    (300, "XV-301", "v5", "Giro (Cilindro D)"),
    (300, "ZSC-301", "c5r", "Sensor Mag. (Cil. D 0°)"),
    (300, "ZSO-301", "c5a", "Sensor Mag. (Cil. D 180°)"),
    (300, "XV-302", "v6", "Vertical (Cilindro E)"),
    (300, "ZSC-302", "c6r", "Sensor Mag. (Cil. E Recuo)"),
    (300, "ZSO-302", "c6a", "Sensor Mag. (Cil. E Avanço)"),
    (300, "VAC-301", "v7", "Válvula de Vácuo"),
    (300, "PIT-301", "p1", "Pressostato de Vácuo"),
    # Setor 400: Termosselagem (6 tags)
    (400, "ZS-400", "s4", "Sensor Capacitivo E4"),
    (400, "XV-401", "v8", "Prensa (Cilindro F)"),
    (400, "ZSC-401", "c7r", "Sensor Mag. (Prensa Recuo)"),
    (400, "ZSO-401", "c7a", "Sensor Mag. (Prensa Avanço)"),
    (400, "HT-401", "h1", "Resistência Cartucho"),
    (400, "TIT-401", "t1", "Sensor de Temperatura"),
    # Setor 500: Ejeção (7 tags)
    (500, "ZS-500", "s5", "Sensor Capacitivo E5"),
    (500, "XV-501", "v9", "Elevador (Cilindro G)"),
    (500, "ZSC-501", "c8r", "Sensor Mag. (Cil. G Recuo)"),
    (500, "ZSO-501", "c8a", "Sensor Mag. (Cil. G Avanço)"),
    (500, "XV-502", "v10", "Extrator (Cilindro H)"),
    (500, "ZSC-502", "c9a", "Sensor Mag. (Cil. H Avanço)"),
    (500, "ZSO-502", "c9r", "Sensor Mag. (Cil. H Recuo)"),
]

assert len(CATALOGO_45_TAGS_ISA51) == 45

# 1) Tabela Hash (dict) para busca exata O(1)
indice_dict = {tag: {"setor": setor, "var": var, "desc": desc} for setor, tag, var, desc in CATALOGO_45_TAGS_ISA51}
assert indice_dict["TIT-401"]["setor"] == 400

# 2) Árvore AVL indexada por chave composta (setor, tag) para consultas ordenadas por faixa
avl_tags = ArvoreAVLHistoriador[Tuple[int, str], Dict[str, Any]]()
for setor, tag, var, desc in CATALOGO_45_TAGS_ISA51:
    avl_tags.inserir((setor, tag), {"setor": setor, "tag": tag, "var": var, "desc": desc})

inv_tags = avl_tags.validar_invariante()
assert inv_tags["valido"] and inv_tags["nos"] == 45

# Consulta ordenada: todas as tags entre Setor 200 e Setor 400 (8 + 9 + 6 = 23 tags)
faixa_200_400 = avl_tags.consultar_intervalo((200, ""), (400, "ZZZ"))
assert len(faixa_200_400) == 23
assert all(200 <= chave[0] <= 400 for chave, _ in faixa_200_400)
assert [k for k, _ in faixa_200_400] == sorted(k for k, _ in faixa_200_400)

print(f"Catálogo AVL validado: {inv_tags['nos']} tags | altura={inv_tags['altura']} | max |BF|={inv_tags['max_abs_bf']}")
print("Nota de engenharia: para busca exata em 45 chaves estáticas, `dict` é O(1);")
print("já a AVL permite consulta ordenada por faixa de setores sem varrer/reordenar todo o catálogo:\n")
print(f"=== Faixa Setor 200 a Setor 400 ({len(faixa_200_400)} tags em ordem lexicográfica por setor) ===")
for (setor, tag), meta in faixa_200_400:
    print(f"  Setor {setor:03d} | {tag:<8} ({meta['var']:<4}) — {meta['desc']}")

---
### 7. Visualização Gráfica (Opcional via `matplotlib`)

A célula abaixo gera os gráficos de:
1. **Desempenho Empírico:** Comparações médias e tempo de busca AVL $\mathcal{O}(\log N)$ vs. Busca Linear $\mathcal{O}(N)$;
2. **Janela RCA do Historiador:** Perfil térmico de `TIT-401` ($^\circ\text{C}$) recuperado via consulta de intervalo *in-order* com destaque para o limiar de $180^\circ\text{C}$ (`HOLD_ESTAGIO_4`).

Caso `matplotlib` não esteja instalado no ambiente local, o bloco é tratado com `try/except ImportError` para manter execução 100% limpa.

In [ ]:
try:
    import matplotlib.pyplot as plt

    ns = [r["N"] for r in resultados_bench]
    comp_avl = [r["comp_avl"] for r in resultados_bench]
    comp_lin = [r["comp_linear"] for r in resultados_bench]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

    # Gráfico 1: Comparações AVL vs Linear (escala log)
    axes[0].plot(ns, comp_lin, marker="o", linewidth=2, color="#d9534f", label="Busca Linear O(N)")
    axes[0].plot(ns, comp_avl, marker="s", linewidth=2, color="#0275d8", label="Árvore AVL O(log₂ N)")
    axes[0].set_yscale("log")
    axes[0].set_title("Historiador Aqua Index: Comparações por Consulta")
    axes[0].set_xlabel("Volume de Eventos Indexados (N)")
    axes[0].set_ylabel("Comparações Médias (escala log)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Gráfico 2: Janela RCA de TIT-401 recuperada via in-order com poda
    t_rca = [(ts - t_base) / 1000.0 for ts, _ in janela_rca]
    temp_rca = [reg.tit_401_c for _, reg in janela_rca]
    axes[1].plot(t_rca, temp_rca, marker="o", linewidth=2, color="#2c7bb6", label="TIT-401 (°C)")
    axes[1].axhline(180.0, linestyle="--", color="#d7191c", linewidth=1.8, label="Limiar R-02 (180 °C)")
    axes[1].set_title("Consulta por Intervalo AVL (RCA — Estação 4)")
    axes[1].set_xlabel("Tempo Relativo Δt (s)")
    axes[1].set_ylabel("Temperatura do Cabeçote (°C)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    print("Gráficos renderizados com sucesso via matplotlib.")
except ImportError:
    print("matplotlib não está instalado neste ambiente — gráficos ignorados (núcleo algorítmico 100% validado).")